<a href="https://colab.research.google.com/github/rkk7452/OVRO-source-matching/blob/main/Animated_PyBDSF_Match_Sources_73MHz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Key Info:

Matching Sources notebook


This notebook relies on:
/content/drive/MyDrive/Caltech SRC '26/combined_sources_73MHz.csv


This file outputs:
/content/drive/MyDrive/Caltech SRC '26/Matching_Source_Pairs_73MHz.mp4
  video of the matching source pair by pair (commented out now, last exported for version 2 of matching)


  Version 3 source matching outputs:

  /content/drive/MyDrive/Caltech SRC '26/73_MHz_Source_Tracking.csv
    csv of full master catalog (for version 3 of matching)
  /content/drive/MyDrive/Caltech SRC '26/73_MHz_Source_Tracking_Detection_log.csv
    csv of each source that was matched (detection log df in version 3)

"""

In [ ]:
# Import necessary packages
import numpy as np
import matplotlib.pyplot as plt
import astropy
from astropy.io import fits
from astropy.wcs import WCS
from astropy.visualization import ZScaleInterval, ImageNormalize
from astropy.coordinates import SkyCoord
import astropy.units as u
import os
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

from matplotlib.ticker import MultipleLocator # custom ticks in plots
import colorcet as cc # more colormaps

In [ ]:
folder_path = "/content/drive/MyDrive/Caltech SRC '26/OVRO-LWA Data/"
ryan_folder_path = "/content/drive/MyDrive/Caltech SRC '26/73MHz_Sources/"
file_name = "combined_sources_73MHz.csv"

In [ ]:
# Needed if running on Google Colab only
# Gives access to data stored on Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


In [ ]:
df = pd.read_csv(ryan_folder_path+file_name)
df = df.sort_values(by='file_name',ascending=True)
#print(df.head())

In [ ]:

#set up
unique_files = df['file_name'].unique()

#unique_files = unique_files[:50]###remove this later, this takes only the first few so the tests run faster!!!

###Set up Matching Hard Thresholds###
seplimit = 3 * u.arcmin # sources can be a max of 3 arc min apart to be considered a match
fluxlimit = 0.2 # 20% limit in flux difference between sources

In [ ]:
#ani.save("/content/drive/MyDrive/Caltech SRC '26/Matching_Source_Pairs_73MHz.mp4", writer='ffmpeg', fps=2)# save to google drive

In [ ]:
"""
Version 3
Algorithm to track and master_id sources across multiple files
"""

deep_int_file_name = "73MHz_Deep_Integration_Sources.csv"

master_catalog = pd.read_csv(os.path.join(ryan_folder_path,deep_int_file_name))

master_catalog.insert(0, 'master_id', range(0, len(master_catalog)))

detections = master_catalog.copy()

master_catalog = master_catalog.set_index("master_id")
master_catalog['num_detections'] = 1
master_catalog = master_catalog.rename(columns={'ra': 'long_ra', 'dec': 'long_dec', 'total_flux': 'long_flux'})
display(master_catalog.head())


master_coords = SkyCoord(ra=master_catalog['long_ra'].values*u.deg, dec=master_catalog['long_dec'].values*u.deg) # first set to match from

#display(detections.head())

for item in unique_files:
  file_2 = df[df['file_name']==item]
  coords_list = SkyCoord(ra=file_2['ra'].values*u.deg, dec=file_2['dec'].values*u.deg) # creats the SkyCoord object from the df
  idx_1, idx_2, d2d, d3d = astropy.coordinates.search_around_sky(master_coords, coords_list, seplimit)
  matches_df = pd.DataFrame({
    'f1_source_id': master_catalog.index[idx_1],

    'f2_source_id': file_2.iloc[idx_2]['source_num'].values,
    'f2_ra': file_2.iloc[idx_2]['ra'].values,
    'f2_dec': file_2.iloc[idx_2]['dec'].values,
    'f2_flux': file_2.iloc[idx_2]['total_flux'].values,

    'separation_arcmin': d2d.to(u.arcmin).value
  })


  #rank and choose only the best new source to match a given source in the master catalog

  matches_df["flux_ratio"] = matches_df["f2_flux"] / master_catalog.loc[matches_df["f1_source_id"], "long_flux"].values #create flux ratio

  #drop matches with higher than fluxlimit flux ratio
  #matches_df = matches_df[(matches_df["flux_ratio"] - 1).abs() <= fluxlimit]

  matches_df["score"] = (matches_df["separation_arcmin"] / seplimit.to(u.arcmin).value)
    #comment out for now b/c flux scales are so different     + 0.5 * np.abs(np.log(matches_df["flux_ratio"]))) #scores a given match:
    #the separation in arc minutes is penalized, where 0 is a perfect match (same location) and 1 is the exact value of seplimit (furthest away possible to be considered a match)
    #the flux ratio is also penzalized. If they have the same flux, the ratio is 1 and log(1) = 0, so there's no penalty
    #then they're added together to create score, in which the lower the score, the better the match

  matches_df = (matches_df.sort_values("score").drop_duplicates(subset="f1_source_id", keep="first").drop_duplicates(subset='f2_source_id',keep='first'))
    # only keeps the best scored match on each end (1-1 mapping)

  #update the master catalog:
  for _, match in matches_df.iterrows():
    master_id = match["f1_source_id"]

    n = master_catalog.loc[master_id, "num_detections"]
    master_catalog.loc[master_id, "num_detections"] = n + 1 # update num detections

  #update the detections df:

  new_rows = []# create empty list

  for _, match in matches_df.iterrows(): # loop through each row in the matches_df
    new_rows.append( {
        "master_id": match['f1_source_id'],
        'source_num': match['f2_source_id'],
        'ra': match['f2_ra'],
        'dec': match['f2_dec'],
        'total_flux': match['f2_flux'],
        'file_name': item
    })

  #add the new_rows list to the detections df
  detections = pd.concat([detections, pd.DataFrame(new_rows)], ignore_index=True)

  #update master_coords
  master_coords = SkyCoord(
    ra=master_catalog["long_ra"].values * u.deg,
    dec=master_catalog["long_dec"].values * u.deg
  )


In [ ]:
#Flux distribution


x = master_catalog['long_flux']
y = df['total_flux']
z = detections['total_flux']

bins = np.linspace(0, 100, 100) # x min, x max, bins

plt.hist(x, bins, alpha=0.5, label='Deep Integration', color='blue')
plt.hist(y, bins, alpha=0.5, label='Short Integration', color='green')
plt.hist(z, bins, alpha=0.3, label='Matches',color='red')


plt.xlabel("Flux")
plt.ylabel("Amount")

plt.legend(loc='upper right')
plt.savefig("/content/drive/MyDrive/Caltech SRC '26/Select_Outputs/73MHz_Flux_Distribution_With_Matches.png")
plt.show()



In [ ]:
#Filter Results

min_detections = 5 #### can change this number ####
min_flux = 0.5 #### can change this number, median was 10.4. Ruby says 1-5 seems good, need to adjust. ####

print(len(master_catalog))

master_catalog = master_catalog[master_catalog['num_detections'] > min_detections].copy()# drop values <= min_detections
master_catalog = master_catalog[master_catalog['long_flux'] > min_flux].copy()# drop values <= min_flux

detections = detections[detections['master_id'].isin(master_catalog.index)].copy() # make sure detections is up to date

#display(master_catalog.head())
#display(master_catalog.tail())

print(len(master_catalog))

In [ ]:
#now...export results!

master_catalog = master_catalog.sort_values(by='num_detections', ascending=False)
display(master_catalog.head())
master_catalog.to_csv("/content/drive/MyDrive/Caltech SRC '26/73_MHz_Source_Tracking_2.csv", index=True, index_label="master_id")
detections.to_csv("/content/drive/MyDrive/Caltech SRC '26/73_MHz_Source_Tracking_Detection_log_2.csv", index = False)

In [ ]:
print(master_catalog['long_flux'].mean())
print(master_catalog['long_flux'].mean())

In [ ]:
#checking results a quick way

import time

countdown = 3
for i in range(countdown):
  print(f'Time before starting.{countdown-i} seconds left...') # I like to go to fullscreen output to watch it
  time.sleep(1)



for master_id, i in master_catalog.head(5).iterrows():

  #set up plot
  plt.figure(figsize=(10,10))
  plt.xlim(df['ra'].min() - 0.5, df['ra'].max() + 0.5)
  plt.ylim(df['dec'].min() - 0.5, df['dec'].max() + 0.5)
  plt.xlabel("RA")
  plt.ylabel("Dec")
  plt.grid(True)

  source = detections[detections["master_id"] == master_id]

  plt.scatter(source["ra"], source["dec"], color="red", s=20, alpha = 0.1) #plots sources detected

  plt.title(f"Master ID {master_id} ({len(source)} detections)")

  plt.show()


In [ ]:
# Check a specific master_id

#checking results a quick way
master_id_to_check = 638

#set up plot
plt.figure(figsize=(10,10))
plt.xlim(df['ra'].min() - 0.5, df['ra'].max() + 0.5)
plt.ylim(df['dec'].min() - 0.5, df['dec'].max() + 0.5)
plt.xlabel("RA")
plt.ylabel("Dec")
plt.grid(True)

source = detections[detections["master_id"] == master_id_to_check]

plt.scatter(source["ra"], source["dec"], color="red", s=20, alpha = 0.1) #plots sources detected

plt.title(f"Master ID {master_id_to_check} ({len(source)} detections)")

plt.show()


In [ ]:
"""
---To Do---

Ruby says:
[done] start with a 20% threshold for failing on flux differences
[done] Do the max difference as 3 arc min for the difference between source positions
[done] maybe consider just cutting out fainter sources if they may be less reliable

New problem: the coordinates are plotted/offset calculated based on ra and dec distance but the plane needs to wrap around so -360 = 0.
  For ex: source w/ Master ID 638

"""

In [ ]:
### Newest Version: Animation + Offset calculations, but separated ###

# This cell is the offset calculation section

"""FYI
combined_offsets is a list
each iteration of the loop, the temporary offset_df is added to the list
after the loop, combined_offsets is converted to the df total_offsets_df

"""



combined_offsets = [] # empty list, will add to it each iteration of the loop


for i in range(0, len(unique_files)):

  file_to_check = unique_files[i] # the name of the nth unique file name in the data

  # create smaller df's with only the sources in the file we're looking at
  detections_sub = detections[detections['file_name']==file_to_check].copy() # detections log subset

  #new smaller df with detections_sub and master catalog based on master id
  merged_sub = detections_sub.merge(
      master_catalog,
      left_on='master_id',
      right_index=True,
      suffixes=('', '_master')
  )

  if merged_sub.empty:
    continue

  offset_df = pd.DataFrame({
      'master_id': merged_sub['master_id'],
      'ra': detections_sub['ra'],
      'dec': detections_sub['dec'],
      'ra_raw_offset': merged_sub['ra'] - merged_sub['long_ra'], #raw difference in degrees
      'dec_offset': merged_sub['dec'] - merged_sub['long_dec'],
      'flux_offset': merged_sub['total_flux'] - merged_sub['long_flux'],
      'file_name': file_to_check
  })

  offset_df['ra_offset'] = offset_df['ra_raw_offset'] * np.cos(np.radians(merged_sub['long_dec'])) # corrected for curvature
  offset_df['offset_dir'] = np.degrees(np.atan2(offset_df['dec_offset'], offset_df['ra_offset'])) # angle of the offset in degrees
  combined_offsets.append(offset_df) # add to combined_offsets


total_offsets_df = pd.concat(combined_offsets, ignore_index=True) # makes it into a df



In [ ]:
display(offset_df.head())

In [ ]:
# Part 2: The animation part, using the offset df created above

#set up

fig, ax = plt.subplots(figsize = (15,10), facecolor = 'black') # black outside background
base_q = ax.quiver([0], [0], [0], [0], [0], cmap=cc.cm['CET_C1'])
base_q.set_clim(-180, 180)

cbar = fig.colorbar(base_q)
cbar.set_label('Offset Direction (Degrees)', color='white')
cbar.ax.yaxis.set_tick_params(color='white')
plt.setp(plt.getp(cbar.ax.axes, 'yticklabels'), color='white')

ax.clear()

#animation loop
def animate (frame_index):
  ax.clear()

  #re apply stylings that are clear by ax.clear()
  ax.set_facecolor('black') # black background inside the plot
  ax.set_xlabel("RA", color='white', fontsize=12) # label axis
  ax.set_ylabel("Dec", color='white', fontsize=12) # label axis
  ax.tick_params(colors='white') # white ticks
  ax.set_xlim(-360, 0) # range of x axis in degrees (RA)
  ax.set_ylim(-90, 90) # range of y axis in degrees (DEC)

  file_to_check = unique_files[frame_index]
  ax.set_title(f"Vector Offsets for {file_to_check}", color='white', fontsize=14) # title plot

  frame_df = total_offsets_df[total_offsets_df['file_name'] == file_to_check]
  if frame_df.empty:
        print(f"Skipping Frame: {frame_index}. File_name: {file_to_check}")
        return

  #magnitudes = np.hypot(offset_df['ra_offset'], offset_df['dec_offset']) # Option 1: color is based off of arrow magnitude
  magnitudes = frame_df['offset_dir'] # Option 2: color is based off of arrow direction

  q = ax.quiver(
      frame_df['ra'],
      frame_df['dec'],
      frame_df['ra_offset'],
      frame_df['dec_offset'],
      magnitudes,
      angles='xy',
      scale_units='xy',
      scale=0.006 ,      # smaller num is larger arrow length
      width=0.002,      # larger num is wider arrow shaft
      headwidth=5,
      headlength=5,
      cmap=cc.cm['CET_C1'])

ani = FuncAnimation(fig, animate, frames=len(unique_files))
ani.save(f'{ryan_folder_path}73MHz_Matching_Vectors.mp4', writer='ffmpeg', fps=2)



In [ ]:
# look into those offset results:
total_offsets_df = total_offsets_df.sort_values(by='ra_offset',ascending=True).copy()
display(total_offsets_df.head())

print(f"Length: {len(total_offsets_df)}")
print('\nOverview')
display(total_offsets_df[['ra_offset', 'dec_offset']].describe())
print("\nskew")
display(total_offsets_df[['ra_offset', 'dec_offset']].skew())
print("\nkurtosis")
display(total_offsets_df[['ra_offset', 'dec_offset']].kurtosis())

In [ ]:
#plotting distribution histograms


#messed up right now because of the skew of 200 or so degree offsets from master id 638
total_offsets_df['ra_offset'].plot.hist(bins=10000, figsize=(10, 6), title="RA Offsets Distribution")
plt.xlabel("Offset (Degrees)")
plt.xlim(-.05,.05)
plt.show()



total_offsets_df['dec_offset'].plot.hist(bins=100, figsize=(10, 6), title="DEC Offsets Distribution")
plt.xlabel("Offset (Degrees)")
plt.show()


In [ ]:
#Plot changes over time

display(total_offsets_df.head())
display(total_offsets_df.tail())
std_ra_by_file = total_offsets_df.groupby('file_name')['ra_offset'].std().rename('Standard_Dev_ra_offset')
std_dec_by_file = total_offsets_df.groupby('file_name')['dec_offset'].std()

mean_dir_by_file = total_offsets_df.groupby('file_name')['offset_dir'].mean()


fig, ax = plt.subplots()

ax.plot(total_offsets_df['file_name'].unique(), std_ra_by_file, color='red')
ax.plot(total_offsets_df['file_name'].unique(), std_dec_by_file, color='green')

ax.set(ylim=(0,0.025))

ax.xaxis.set_major_locator(MultipleLocator(20))
ax.set_xlabel('Time')
ax.tick_params(labelbottom=False)


"""
ax.plot(total_offsets_df['file_name'].unique(), mean_dir_by_file, color='blue')

ax.set(ylim=(-90, 90))
ax.xaxis.set_major_locator(MultipleLocator(20))
ax.set_ylabel('Mean Offset Direction per Image (deg)')
plt.title('73 MHz Mean Offset Direction over time on 2025.05.07')
"""

plt.show()

In [ ]:
std_ra_by_file = std_ra_by_file.sort_values(ascending=False).copy()
print("sorted!")
display(std_ra_by_file.head())

In [ ]:
#Plot specific figures with vectors

file_to_check = "73MHz-Clean-Snapshot-20250507_103142-image.fits"

# create smaller df's with only the sources in the file we're looking at
detections_sub = detections[detections['file_name']==file_to_check].copy() # detections log subset

#new smaller df with detections_sub and master catalog based on master id
merged_sub_new = detections_sub.merge(
    master_catalog,
    left_on='master_id',
    right_index=True,
    suffixes=('', '_master')
)

#print(f'Merged Detections & Catalog Subset. Len: {len(merged_sub_new)}') # check
#display(merged_sub_new.head())

offset_df_new = pd.DataFrame({
    'master_id': merged_sub_new['master_id'],
    'ra_raw_offset': merged_sub_new['ra'] - merged_sub_new['long_ra'], #raw difference in degrees
    'dec_offset': merged_sub_new['dec'] - merged_sub_new['long_dec'],
    'flux_offset': merged_sub_new['total_flux'] - merged_sub_new['long_flux'],
})

offset_df_new['ra_offset'] = offset_df_new['ra_raw_offset'] * np.cos(np.radians(merged_sub_new['long_dec'])) # corrected for curvature
offset_df_new['offset_dir'] = np.degrees(np.atan2(offset_df_new['dec_offset'], offset_df_new['ra_offset'])) # angle of the offset in degrees



offset_df_new['file_name'] = file_to_check # know which file it came from

#print(f'offset_df_new. Len: {len(offset_df_new)}')
#display(offset_df_new.head())
#display(offset_df_new.tail())


#plot with vectors:

fig, ax = plt.subplots(figsize = (15,10), facecolor = 'black') # black outside background
ax.set_facecolor('black') # black background inside the plot
ax.set_xlabel("RA", color='white', fontsize=12) # label axis
ax.set_ylabel("Dec", color='white', fontsize=12) # label axis
ax.tick_params(colors='white') # white ticks
ax.set_title(f"Vector Offsets for {file_to_check}", color='white', fontsize=14) # title plot


ax.set_xlim(-360, 0) # range of x axis in degrees (RA)
ax.set_ylim(-90, 90) # range of y axis in degrees (DEC)


# magnitudes = np.hypot(offset_df_new['ra_offset'], offset_df_new['dec_offset']) # old, color based off of magnitude of arrows
magnitudes = offset_df_new['offset_dir']
q = ax.quiver(detections_sub['ra'],
              detections_sub['dec'],
              offset_df_new['ra_offset'],
              offset_df_new['dec_offset'],
              magnitudes,
              angles='xy',
              scale_units='xy',
              scale=0.006 ,      # smaller num is larger arrow length
              width=0.002,      # larger num is wider arrow shaft
              headwidth=5,
              headlength=5,
              cmap=cc.cm.CET_C8)

cbar = fig.colorbar(q) # make a legend color bar
cbar.set_label('Offset Magnitude (Degrees)', color='white')
cbar.ax.yaxis.set_tick_params(color='white')
plt.setp(plt.getp(cbar.ax.axes, 'yticklabels'), color='white')



plt.show()

In [ ]:
#look at longer integrated sources rather than just mean for cross matching of sources
#will have files later today to look at

#make a movie of the source match vector graphs